# AIM Tutorial — Qiskit Port

This is the Qiskit-side companion to `aim_tutorial.ipynb`. Same physics, same
walkthrough structure — build the Anderson Impurity Model, solve for the ground
state, compute the Green's function, compare against exact diagonalization — but
every quantum-circuit step now goes through Qiskit instead of Qulacs, via the
`qiskit_port` package.

**Package layout** (see `qiskit_port_next_steps.pptx` for the full architecture):

`
qiskit_utils.py   -- shared low-level helpers
qiskit_ansatz.py  -- circuit-building only
qiskit_vqe.py     -- optimizer, Lanczos, Green's function (complete)
dmft_qiskit.py    -- high-level driver (this notebook calls into it)
`

Each step below is shown individually first — mirroring the original tutorial's
teaching style — with the one-line `run_aim_qiskit()` version at the end.

## 0. Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import exact
import vqe
import qiskit_port.qiskit_vqe as qv
import qiskit_port.dmft_qiskit as dmft_qiskit
from qiskit_port.qiskit_utils import qulacs_to_python_ordering_qiskit

## 1. Initialize the AIM system

Builds the model, the connectivity graphs, the spin-up/spin-down qubit indices,
the impurity orbital, and the qubit Hamiltonian. This step is **backend-agnostic**
— reused directly from `dmft.py`, no Qiskit-specific version needed.

In [2]:
SYSTEM_SIZE = 4
SEED = 0

(
    impurity_orbital,
    test_model,
    n_orbitals,
    up_qubit_indices,
    down_qubit_indices,
    connected_graphs,
    qubit_hamiltonian,
) = dmft_qiskit.initialize_system(SYSTEM_SIZE, SEED)

print(f"System built: impurity_orbital={impurity_orbital}, n_orbitals={n_orbitals}")
print(f"up_qubit_indices={up_qubit_indices}, down_qubit_indices={down_qubit_indices}")

System built: impurity_orbital=2, n_orbitals=4
up_qubit_indices=[0, 1, 2, 3], down_qubit_indices=[4, 5, 6, 7]


## 2. Solve the exact ground state

Classical exact diagonalization — no VQE, no optimizer, no circuits. This is the
ground-truth reference everything else gets compared against.

In [3]:
exact_energy, exact_state, exact_charge, exact_spin, degenerate, exact_gs_record = (
    exact.solve_exact_gs(test_model)
)

print(f"Exact ground state: energy={exact_energy:.6f}, charge={exact_charge}, "
      f"spin={exact_spin}, degenerate={degenerate}")

Exact ground state: energy=-6.342495, charge=4.0, spin=0.0, degenerate=False


## 3. Solve the Qiskit VQE ground state

This is the piece that used to require reusing Qulacs's optimized angles. As of
this project's Qiskit port, it's fully independent: `solve_vqe_qiskit()` searches
every allowed charge/spin sector and optimizes the ansatz energy in each one,
entirely on the Qiskit side.

In [ ]:
VQE_DEPTH = 1
OPTIMIZER = "COBYLA"
GS_GTOL = 5e-5

gs_result = dmft_qiskit.calculate_ground_state_qiskit(
    test_model=test_model,
    qubit_hamiltonian=qubit_hamiltonian,
    connected_graphs=connected_graphs,
    up_qubit_indices=up_qubit_indices,
    down_qubit_indices=down_qubit_indices,
    vqe_depth=VQE_DEPTH,
    optimizer=OPTIMIZER,
    gs_gtol=GS_GTOL,
    display=True,
)

print(f"\nQiskit VQE ground state: energy={gs_result['energy']:.6f}, "
      f"charge={gs_result['charge']}, spin={gs_result['spin']}")

Sector: n_electrons=0, z_spin=0, init_occ=[]
  -> energy = -0.000000, success = True
Sector: n_electrons=1, z_spin=-1, init_occ=[4]


## 4. Compare ground states

Two numbers matter here: the **energy relative error** and the **state fidelity**
against the exact ground state. This is the same comparison that gave the
~4-parts-per-million energy agreement earlier this project — independently solved,
no shared angles between the two backends.

In [ ]:
gs_compare = dmft_qiskit.compare_ground_states_qiskit(
    exact_energy=exact_energy,
    qiskit_energy=gs_result["energy"],
    exact_state=exact_state,
    qiskit_state_ordered=gs_result["state_ordered"],
)

print(f"Energy relative error: {gs_compare['gs_energy_rel_error']:.3e}")
print(f"State fidelity:        {gs_compare['gs_fidelity']:.6f}")
print(f"gs_error (1 - overlap): {gs_compare['gs_error']:.3e}")

## 5. Compute the Green's function

Builds the particle-removal/addition Krylov states, derives their charge/spin
sectors, and runs the variational Lanczos process on each branch via
`calculate_gf_vqe_qiskit()` — which already has the validated removal/addition
sign convention built in (`w=-w` for removal, combined via subtraction).

In [ ]:
W_MIN, W_MAX, N_W, ETA = -25.0, 25.0, 1000, 0.1
w = np.linspace(W_MIN, W_MAX, N_W, dtype=np.complex128) + 1j * ETA

CONV_TOL = 1e-5
MAXITER = 5000
GF_GTOL = 5e-5

qiskit_gf = dmft_qiskit.calculate_green_function_qiskit(
    impurity_orbital=impurity_orbital,
    qubit_hamiltonian=qubit_hamiltonian,
    ground_state_result=gs_result,
    up_qubit_indices=up_qubit_indices,
    down_qubit_indices=down_qubit_indices,
    connected_graphs=connected_graphs,
    n_orbitals=n_orbitals,
    vqe_depth=VQE_DEPTH,
    w=w,
    optimizer=OPTIMIZER,
    conv_tol=CONV_TOL,
    maxiter=MAXITER,
    gf_gtol=GF_GTOL,
)

print(f"Removal Krylov dim: {qiskit_gf['krylov_dim_minus']}, "
      f"Addition Krylov dim: {qiskit_gf['krylov_dim_plus']}")
print(f"Sectors: {qiskit_gf['sectors']}")

## 6. Compute the exact Green's function, and compare

The exact Green's function reuses the **same Krylov dimensions** the Qiskit VQE run
used — same truncation depth on both sides, for a fair comparison, matching the
original `dmft.py`'s behavior exactly.

In [ ]:
new_phi_plus_ordered = qulacs_to_python_ordering_qiskit(
    qiskit_gf["phi_plus_array"], gs_result["n_qubits"]
)
new_phi_minus_ordered = qulacs_to_python_ordering_qiskit(
    qiskit_gf["phi_minus_array"], gs_result["n_qubits"]
)

g_exact, exact_gf_record = exact.calculate_gf_exact(
    exact_gs=exact_state,
    test_model=test_model,
    new_phi_plus=new_phi_plus_ordered,
    new_phi_minus=new_phi_minus_ordered,
    n_orbitals=n_orbitals,
    w=w,
    exact_gs_energy=exact_energy,
    gs=False,
    vqe_krylov_ideal_dim_plus=qiskit_gf["krylov_dim_plus"],
    vqe_krylov_ideal_dim_minus=qiskit_gf["krylov_dim_minus"],
    impurity_orbital=impurity_orbital,
)
spectral_exact = -(1.0 / np.pi) * np.imag(g_exact)

rel_error, rel_error_record = dmft_qiskit.calculate_relative_errors(
    qiskit_gf["g_total"], g_exact
)
print(f"Green's function relative error (Qiskit vs exact): {rel_error:.3e}")
print(f"Average pointwise difference: {rel_error_record['g_avg_rel_diff']:.3e}")

In [ ]:
imag_g = np.imag(g_total)
spectral = -(1.0 / np.pi) * imag_g

print("System qubits:", n_qubits)
print("min Im[G]:", np.min(imag_g))
print("max Im[G]:", np.max(imag_g))
print("min A(w):", np.min(spectral))
print("number of negative A values:", np.count_nonzero(spectral < -1e-10))

## 7. Plot

In [ ]:
dmft_qiskit.plot_green_functions(w, qiskit_gf["g_total"], g_exact, impurity_orbital)

In [ ]:
dmft_qiskit.plot_spectral_functions(w, qiskit_gf["spectral"], spectral_exact)

## 8. Or, do all of the above in one call

Everything above — steps 1 through 7 — is exactly what `run_aim_qiskit()` does
internally. This is the version a notebook user should actually reach for; the
step-by-step breakdown above is for understanding what's happening underneath it.

In [ ]:
results = dmft_qiskit.run_aim_qiskit(
    system_size=SYSTEM_SIZE,
    seed=SEED,
    vqe_depth=VQE_DEPTH,
    optimizer=OPTIMIZER,
    gs_gtol=GS_GTOL,
    gf_gtol=GF_GTOL,
    conv_tol=CONV_TOL,
    maxiter=MAXITER,
    display=True,
    plot=True,
)

print("\nTop-level keys:", list(results.keys()))
print("Ground-state comparison:", results["ground_state"]["comparison"])
print("Green's-function relative error:", results["errors"]["g_rel_error"])

In [ ]:
"""Simple Qulacs vs Qiskit comparison: ground state + Green's function."""

import numpy as np
import matplotlib.pyplot as plt

# --- required compatibility patches ---
import anderson_impurity_model as aim
from scipy.optimize import minimize as _real_minimize, OptimizeResult
from scipy._lib._util import _RichResult

def _minimize_safe_maxiter(fun, x0, args=(), method=None, **kwargs):
    if kwargs.get("options", {}).get("maxiter") is not None:
        kwargs["options"]["maxiter"] = int(kwargs["options"]["maxiter"])
    return _real_minimize(fun, x0, args=args, method=method, **kwargs)

_OPTIONAL_FIELDS = {"nit": 0, "njev": 0, "nhev": 0, "jac": None, "hess": None, "hess_inv": None}

def _safe_get(self, name):
    try:
        return dict.__getitem__(self, name)
    except KeyError:
        if name in _OPTIONAL_FIELDS:
            return _OPTIONAL_FIELDS[name]
        raise AttributeError(name)

def _safe_del(self, name):
    try:
        dict.__delitem__(self, name)
    except KeyError:
        pass

aim.minimize = _minimize_safe_maxiter
_RichResult.__getattr__ = _safe_get
_RichResult.__getitem__ = lambda self, name: _safe_get(self, name)
OptimizeResult.__delattr__ = _safe_del

# save_result()/load_all_results() are how symmetric_ansatz_test()
# collects its own sector results -- it writes each one to results_file
# during the search, then re-reads the WHOLE file back afterward to
# build sector_to_energy_dict. That's real file I/O with real failure
# modes (bad path, stale kernel patches, permissions). Replace both
# with a matched in-memory pair instead: no disk touched at all, so
# none of those failure modes can happen. Because this fully replaces
# the functions rather than wrapping/capturing "the original," it's
# safe to re-run this cell any number of times -- each run just starts
# the in-memory store fresh.
_in_memory_store = {}

def _save_result_mem(spin_charge, optimize_result, qulacs_spin_charge, results_file):
    _in_memory_store.setdefault(results_file, []).append({spin_charge: (optimize_result, qulacs_spin_charge)})

def _load_all_results_mem(filename):
    for item in _in_memory_store.get(filename, []):
        yield item

def _save_checkpoint_mem(x, fun, iteration, spin_charge, checkpoint_file):
    pass  # nothing in this code path ever reads checkpoints back -- see load_checkpoint below

aim.save_result = _save_result_mem
aim.load_all_results = _load_all_results_mem
aim.save_checkpoint = _save_checkpoint_mem

import dmft
import vqe
import qiskit_port.dmft_qiskit as dmft_qiskit

# --- build system ---
SYSTEM_SIZE = 3
SEED = 0
VQE_DEPTH = 1
OPTIMIZER = "COBYLA"
GS_GTOL = 5e-5
GF_GTOL = 5e-5
CONV_TOL = 1e-5
MAXITER = 5000

w = np.linspace(-25, 25, 1000, dtype=np.complex128) + 1j * 0.1  # widened to match the exact-reference plots' full range -- the narrower +-10 window was missing the smaller peak near +-16

(impurity_orbital, test_model, n_orbitals, up_qubit_indices, down_qubit_indices,
 connected_graphs, qubit_hamiltonian) = dmft.initialize_system(SYSTEM_SIZE, SEED)

# --- Qulacs ---
# results_file/checkpoint_file are now just dict keys in the in-memory
# store above, not real paths -- any string works, doesn't need to
# exist or be writable.
qulacs_energy, qulacs_charge, qulacs_spin, qulacs_angles, qulacs_record = vqe.solve_vqe(
    test_model=test_model, vqe_depth=VQE_DEPTH, optimizer=OPTIMIZER, gs_gtol=GS_GTOL,
    checkpoint_file="qulacs_run", results_file="qulacs_run", display=False,
)
qulacs_ham, qulacs_H2, n_qubits = vqe.create_qulacs_hamiltonian(qubit_hamiltonian, qulacs_energy)
qulacs_gs, _ = vqe.construct_vqe_gs(
    n_qubits=n_qubits, qulacs_ham=qulacs_ham, up_qubit_indices=up_qubit_indices,
    down_qubit_indices=down_qubit_indices, vqe_depth=VQE_DEPTH, connected_graphs=connected_graphs,
    minimum_angles=qulacs_angles, vqe_nu=qulacs_record["vqe_nu"], vqe_nd=qulacs_record["vqe_nd"],
)
(qulacs_phi_minus, _, qulacs_phi_minus_norm, qulacs_phi_plus, _, qulacs_phi_plus_norm, _) = (
    vqe.vqe_krylov_zero_state(vqe_gs=qulacs_gs, impurity_orbital=impurity_orbital, n_qubits=n_qubits)
)
(_, _, cm, sm, dm, _, _, cp, sp, dp, _) = vqe.construct_vqe_krylov_dimensions(
    up_idx=up_qubit_indices, impurity_orbital=impurity_orbital, n_orbitals=n_orbitals,
    vqe_charge=qulacs_charge, vqe_spin=qulacs_spin,
)
g_qulacs, _ = vqe.calculate_gf_vqe(
    qulacs_ham=qulacs_ham, qulacs_ham_2=qulacs_H2, phi_minus=qulacs_phi_minus, phi_plus=qulacs_phi_plus,
    vqe_charge_minus=cm, vqe_charge_plus=cp, vqe_spin_minus=sm, vqe_spin_plus=sp,
    up_qubit_indices=up_qubit_indices, down_qubit_indices=down_qubit_indices,
    connected_graphs=connected_graphs, w=w, vqe_krylov_ideal_dim_minus=dm, vqe_krylov_ideal_dim_plus=dp,
    phi_minus_norm=qulacs_phi_minus_norm, phi_plus_norm=qulacs_phi_plus_norm, vqe_depth=VQE_DEPTH,
    optimizer=OPTIMIZER, conv_tol=CONV_TOL, maxiters=MAXITER, gf_gtol=GF_GTOL,
)
spectral_qulacs = -(1.0 / np.pi) * np.imag(g_qulacs)
print(f"Qulacs: energy={qulacs_energy:.6f}")

# --- Qiskit ---
qiskit_gs = dmft_qiskit.calculate_ground_state_qiskit(
    test_model=test_model, qubit_hamiltonian=qubit_hamiltonian, connected_graphs=connected_graphs,
    up_qubit_indices=up_qubit_indices, down_qubit_indices=down_qubit_indices, vqe_depth=VQE_DEPTH,
    optimizer=OPTIMIZER, gs_gtol=GS_GTOL, display=False,
)
qiskit_gf = dmft_qiskit.calculate_green_function_qiskit(
    impurity_orbital=impurity_orbital, qubit_hamiltonian=qubit_hamiltonian, ground_state_result=qiskit_gs,
    up_qubit_indices=up_qubit_indices, down_qubit_indices=down_qubit_indices, connected_graphs=connected_graphs,
    n_orbitals=n_orbitals, vqe_depth=VQE_DEPTH, w=w, optimizer=OPTIMIZER, conv_tol=CONV_TOL,
    maxiter=MAXITER, gf_gtol=GF_GTOL,
)
g_qiskit = qiskit_gf["g_total"]
spectral_qiskit = qiskit_gf["spectral"]
print(f"Qiskit: energy={qiskit_gs['energy']:.6f}")

# --- compare ---
print(f"Energy diff: {abs(qulacs_energy - qiskit_gs['energy']):.3e}")

plt.figure(figsize=(9, 5))
plt.plot(w.real, spectral_qiskit, label="Qiskit")
plt.plot(w.real, spectral_qulacs, "--", label="Qulacs")
plt.xlabel(r"$\omega$")
plt.ylabel(r"$A(\omega)$")
plt.legend()
plt.title("Qiskit vs Qulacs Spectral Function")
plt.tight_layout()
plt.show()

# --- compare peaks, not just the raw curve ---
# A raw max/mean diff over the whole curve can be misleading (one
# steep shoulder near a pole dominates it even when the physics
# matches well). Peak position/height is the physically meaningful
# quantity -- same approach validated earlier in this project.
from scipy.signal import find_peaks

def get_main_peaks(omega, spectral, prominence=1e-3):
    peak_indices, _ = find_peaks(spectral, prominence=prominence)
    peak_indices = peak_indices[np.argsort(omega[peak_indices])]
    return [(float(omega[i]), float(spectral[i])) for i in peak_indices]

def match_peaks_by_position(peaks_a, peaks_b, max_delta=1.0):
    remaining_b = list(peaks_b)
    matches, unmatched_a = [], []
    for pa in peaks_a:
        if not remaining_b:
            unmatched_a.append(pa)
            continue
        idx = min(range(len(remaining_b)), key=lambda i: abs(remaining_b[i][0] - pa[0]))
        pb = remaining_b[idx]
        if abs(pa[0] - pb[0]) <= max_delta:
            matches.append((pa, pb, pa[0] - pb[0]))
            remaining_b.pop(idx)
        else:
            unmatched_a.append(pa)
    return matches, unmatched_a, remaining_b

qiskit_peaks = get_main_peaks(w.real, spectral_qiskit)
qulacs_peaks = get_main_peaks(w.real, spectral_qulacs)

print(f"\nQiskit peaks ({len(qiskit_peaks)}):")
for om, height in qiskit_peaks:
    print(f"  omega={om:.6f}, A={height:.6f}")

print(f"\nQulacs peaks ({len(qulacs_peaks)}):")
for om, height in qulacs_peaks:
    print(f"  omega={om:.6f}, A={height:.6f}")

matches, unmatched_qiskit, unmatched_qulacs = match_peaks_by_position(qiskit_peaks, qulacs_peaks)
print(f"\nMatched peaks ({len(matches)}):")
for (om_a, h_a), (om_b, h_b), delta in matches:
    height_pct_diff = abs(h_a - h_b) / h_b * 100 if h_b else float("nan")
    print(f"  Qiskit omega={om_a:.6f} (A={h_a:.4f})  <->  "
          f"Qulacs omega={om_b:.6f} (A={h_b:.4f})   "
          f"delta_omega={delta:.6f}, height diff={height_pct_diff:.2f}%")

if unmatched_qiskit:
    print(f"\n{len(unmatched_qiskit)} Qiskit peak(s) with no Qulacs match: {unmatched_qiskit}")
if unmatched_qulacs:
    print(f"{len(unmatched_qulacs)} Qulacs peak(s) with no Qiskit match: {unmatched_qulacs}")